In [ ]:
!pip install -q pandas pyarrow regex langdetect transformers==4.41.2

In [ ]:
import html
import re
import unicodedata
import hashlib
from pathlib import Path

import pandas as pd
from langdetect import detect, DetectorFactory
from transformers import AutoTokenizer

DetectorFactory.seed = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PRE_MONTHLY_DIR = Path("/content/drive/MyDrive/news_data/pre_2022")
POST_MONTHLY_DIR = Path("/content/drive/MyDrive/news_data/post_2021")

PRE_PROCESSED_DIR = Path("/content/drive/MyDrive/news_data/pre_2022_processed")
POST_PROCESSED_DIR = Path("/content/drive/MyDrive/news_data/post_2021_processed")

PRE_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
POST_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Binoculars uses Falcon tokenizer consistency and truncates to max_token_observed.
# Your report says use 256-token chunks.
TOKENIZER_NAME = "tiiuae/falcon-7b"
MAX_TOKENS = 256

# Filters
MIN_WORDS = 100
MAX_WORDS = 2000
MIN_SECTION_CHARS = 80

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
ZERO_WIDTH_RE = re.compile(r"[\u200B-\u200D\uFEFF]")
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
WHITESPACE_RE = re.compile(r"\s+")

def clean_for_binoculars(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # HTML entities -> characters
    text = html.unescape(text)

    # Normalize unicode while preserving visible punctuation/case
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible unicode
    text = ZERO_WIDTH_RE.sub("", text)

    # Remove bare URLs
    text = URL_RE.sub("", text)

    # Normalize whitespace only
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text

In [ ]:
def safe_detect_english(text: str) -> bool:
    if not text or len(text) < 40:
        return False
    try:
        return detect(text) == "en"
    except Exception:
        return False

def word_count(text: str) -> int:
    return len(text.split())

def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()

In [ ]:
SECTION_SPLIT_RE = re.compile(r"\n\s*\n+")

def split_into_sections(text: str) -> list[str]:
    # Preserve paragraph structure if present
    raw_sections = [s.strip() for s in SECTION_SPLIT_RE.split(text) if s.strip()]

    # Fallback: if the text has no paragraph breaks, use the whole text
    if not raw_sections:
        raw_sections = [text.strip()]

    # Drop tiny sections
    sections = [s for s in raw_sections if len(s) >= MIN_SECTION_CHARS]

    if not sections:
        sections = [text.strip()]

    return sections

def make_one_chunk_per_section(section_text: str, tokenizer, max_tokens: int = 256) -> list[str]:
    """
    Returns 0 or 1 chunk for this section.
    - If section is <= max_tokens: keep it as a single chunk
    - If section is longer: take the first max_tokens tokens
    """
    ids = tokenizer.encode(
        section_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_tokens
    )

    if len(ids) == 0:
        return []

    chunk_ids = ids

    chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True).strip()

    return [chunk_text] if chunk_text else []

In [ ]:
def preprocess_monthly_news_file(
    input_parquet: Path,
    output_parquet: Path,
    tokenizer,
    max_tokens: int = 256,
    min_words: int = 100,
    max_words: int = 2000,
):
    df = pd.read_parquet(input_parquet)

    required_cols = ["maintext"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{input_parquet} is missing required column: {col}")

    rows = []
    seen_chunk_hashes = set()

    for idx, row in df.iterrows():
        raw_text = row.get("maintext", "")
        cleaned = clean_for_binoculars(raw_text)

        if not cleaned:
            continue

        wc = word_count(cleaned)
        if wc < min_words or wc > max_words:
            continue

        if not safe_detect_english(cleaned):
            continue

        sections = split_into_sections(cleaned)

        for section_idx, section in enumerate(sections):
            chunks = make_one_chunk_per_section(
                section_text=section,
                tokenizer=tokenizer,
                max_tokens=max_tokens,
            )

            for chunk_idx, chunk_text in enumerate(chunks):
                chunk_text = clean_for_binoculars(chunk_text)

                if not chunk_text:
                    continue

                chunk_wc = word_count(chunk_text)
                if chunk_wc == 0:
                    continue

                chunk_ids = tokenizer.encode(chunk_text, add_special_tokens=False)
                n_tokens = len(chunk_ids)

                # Keep exact-dedup at chunk level
                h = text_hash(chunk_text)
                if h in seen_chunk_hashes:
                    continue
                seen_chunk_hashes.add(h)

                rows.append({
                    "source_file": input_parquet.name,
                    "article_row_idx": idx,
                    "url": row.get("url", None),
                    "source_domain": row.get("source_domain", None),
                    "date_publish": row.get("date_publish", None),
                    "time_bucket": row.get("time_bucket", None),
                    "month_key": row.get("month_key", None),
                    "section_idx": section_idx,
                    "chunk_idx": chunk_idx,
                    "chunk_text": chunk_text,
                    "word_count": chunk_wc,
                    "n_tokens": n_tokens,
                    "chunk_hash": h,
                })

    out_df = pd.DataFrame(rows)

    if len(out_df) > 0:
        out_df = out_df.drop_duplicates(subset=["chunk_hash"]).reset_index(drop=True)

    out_df.to_parquet(output_parquet, index=False)

    print(f"Saved {len(out_df)} chunks to {output_parquet}")
    return out_df

In [ ]:
def process_monthly_folder(
    input_dir: Path,
    output_dir: Path,
    tokenizer,
    max_tokens: int = 256,
):
    input_files = sorted(input_dir.glob("*.parquet"))

    for input_path in input_files:
        output_name = input_path.stem + "_binoculars_ready.parquet"
        output_path = output_dir / output_name

        if output_path.exists():
            print(f"Skipping {input_path.name} (already processed)")
            continue

        try:
            preprocess_monthly_news_file(
                input_parquet=input_path,
                output_parquet=output_path,
                tokenizer=tokenizer,
                max_tokens=max_tokens,
                min_words=MIN_WORDS,
                max_words=MAX_WORDS,
            )
        except Exception as e:
            print(f"Failed on {input_path.name}: {e}")

In [ ]:
process_monthly_folder(
    input_dir=PRE_MONTHLY_DIR,
    output_dir=PRE_PROCESSED_DIR,
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
)

In [ ]:
process_monthly_folder(
    input_dir=POST_MONTHLY_DIR,
    output_dir=POST_PROCESSED_DIR,
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
)

In [ ]:
!git clone https://github.com/ahans30/Binoculars.git
%cd Binoculars
!pip install -e .

In [ ]:
import torch

from binoculars import Binoculars

bino = Binoculars()

In [ ]:
sample_processed = pd.read_parquet(
    PRE_PROCESSED_DIR / "pre_2022_2019-05_binoculars_ready.parquet"
)

texts = sample_processed["chunk_text"].tolist()

scores = bino.compute_score(texts[:32])
preds = bino.predict(texts[:32])

scored_df = sample_processed.iloc[:32].copy()
scored_df["binoculars_score"] = scores
scored_df["binoculars_pred"] = preds

scored_df.head()

In [ ]:
post_sample_processed = pd.read_parquet(
    POST_PROCESSED_DIR / "post_2021_2025-03_binoculars_ready.parquet"
)

post_texts = post_sample_processed["chunk_text"].tolist()

post_scores = bino.compute_score(post_texts)

post_scored_df = post_sample_processed.copy()
post_scored_df["binoculars_score"] = post_scores

post_scored_df.head()

from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/news_data/scored_outputs")

file_path = SAVE_DIR / f"scored_df_2025-03.parquet"

post_scored_df.to_parquet(file_path, index=False)

print("Saved to:", file_path)

In [ ]:
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/news_data/scored_outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

file_path = SAVE_DIR / f"scored_df_2022_2019-05.parquet"

scored_df.to_parquet(file_path, index=False)

print("Saved to:", file_path)

In [ ]:
print(scored_df)